# Resultados finales: predicción paralela de mapas de calor (LST) en Tlaxcala y Puebla

**Tercer avance.** Se entrena con 2016–2023 y se predicen los **12 mapas de 2024**, un año que el modelo nunca vio, y se compara contra tres referencias sencillas.

**Paralelismo (multiprocessing, sin joblib ni n_jobs):**
1. Construcción de la tabla pixel-mes con memoria compartida (paralelismo de datos).
2. Búsqueda de hiperparámetros de KNN, RF y SVR (paralelización exploratoria, reparto estático vs. dinámico).
3. Predicción de los mapas de 2024 por bloques de pixeles (paralelismo de datos).

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"          # cada proceso usa un solo hilo: el paralelismo lo controlamos nosotros

import ast
import time
import multiprocessing as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import datos
import modelos
import paralelo as par

RUTA_LST = os.environ.get("RUTA_LST", "LST_mensual.tif")
RUTA_NDVI = os.environ.get("RUTA_NDVI", "NDVI_mensual.tif")
RUTA_ALT = os.environ.get("RUTA_ALT", "ALTITUD.tif")
ANIO_PRUEBA = 2024
N_MAX_ENTRENAMIENTO = int(os.environ.get("N_MAX", 40000))
REPETIR_BUSQUEDA = False   # True = volver a correr las 46 combinaciones (≈40 min con 8 núcleos)
NUCLEOS = mp.cpu_count()
LISTA_PROCESOS = [p for p in [1, 2, 4, 8, 16] if p <= NUCLEOS]

os.makedirs("resultados", exist_ok=True)
os.makedirs("figuras", exist_ok=True)
COLORES = {"KNN": "#2a78d6", "RF": "#eb6834", "SVR": "#1baf7a",
           "Persistencia_t1": "#9a9a9a", "Persistencia_t12": "#6b6b6b", "Climatologia": "#3d3d3d"}
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
print("Núcleos:", NUCLEOS, "| Procesos a medir:", LISTA_PROCESOS)

## 1. Lectura de los mapas mensuales

In [ ]:
mapas = datos.leer_mapas(RUTA_LST, RUTA_NDVI, RUTA_ALT)
lst, ndvi, alt, lat, lon = mapas
FORMA = alt.shape
EXTENSION = [lon[0], lon[-1], lat[-1], lat[0]]

## 2. Construcción de la tabla en paralelo: versión del 2º avance vs. versión final

En el 2º avance, 8 procesos tardaban **más** que 1: cada proceso recibía una copia de los mapas y regresaba su bloque de la tabla serializado, y unir los bloques costaba más que armarlos.

La versión final (`datos.construir_variables`) pone los mapas y la tabla de salida en **memoria compartida**:
- **Fase 1:** cada proceso cuenta los registros válidos de su bloque de filas.
- **Suma prefija:** el proceso principal calcula dónde escribe cada bloque.
- **Fase 2:** cada proceso escribe sus registros directamente en la tabla compartida.

Se verifica que las dos versiones producen exactamente la misma tabla.

In [ ]:
tiempos_v1, tiempos_v2 = {}, {}
for p in LISTA_PROCESOS:
    df_v1, tiempos_v1[p] = datos.construir_variables_v1(mapas, procesos=p)
    df, tiempos_v2[p] = datos.construir_variables(mapas, procesos=p)
    print(f"{p:>2} procesos:  v1 {tiempos_v1[p]:6.2f} s   v2 {tiempos_v2[p]:6.2f} s   ({len(df):,} filas)")

# Las dos versiones deben dar la misma tabla
cols = list(df_v1.columns)
dif = max(float(np.abs(df_v1[c].to_numpy(float) - df[c].to_numpy(float)).max()) for c in cols)
print("Misma tabla:", df_v1.shape == df[cols].shape, "| diferencia máxima =", dif)
del df_v1

tabla_v1 = par.metricas_paralelas(tiempos_v1).assign(version="v1 (2º avance)")
tabla_v2 = par.metricas_paralelas(tiempos_v2).assign(version="v2 (memoria compartida)")
construccion = pd.concat([tabla_v1, tabla_v2], ignore_index=True)
construccion.to_csv("resultados/construccion_tabla_v1_vs_v2.csv", index=False)
construccion.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for t, color in ((tabla_v1, "#9a9a9a"), (tabla_v2, "#2a78d6")):
    axes[0].plot(t["procesos"], t["tiempo_s"], marker="o", lw=2, color=color, label=t["version"][0])
    axes[1].plot(t["procesos"], t["speedup"], marker="o", lw=2, color=color, label=t["version"][0])
axes[1].plot(LISTA_PROCESOS, LISTA_PROCESOS, ls="--", color="#bbbbbb", label="Ideal")
axes[0].set_title("Construcción de la tabla: tiempo"); axes[0].set_ylabel("Segundos")
axes[1].set_title("Construcción de la tabla: speedup"); axes[1].legend(frameon=False)
for ax in axes:
    ax.set_xlabel("Número de procesos"); ax.set_xticks(LISTA_PROCESOS)
axes[0].legend(frameon=False)
fig.tight_layout(); fig.savefig("figuras/construccion_tabla_v1_vs_v2.png", dpi=200); plt.show()

In [ ]:
resumen_datos = pd.DataFrame({"dato": ["Filas (pixel-mes)", "Pixeles con datos", "Meses", "Años"],
    "valor": [len(df), df.groupby(["fila", "col"]).ngroups, df[["anio", "mes"]].drop_duplicates().shape[0],
              f"{df['anio'].min()}–{df['anio'].max()}"]})
resumen_datos.to_csv("resultados/resumen_datos.csv", index=False)
resumen_datos

## 3. Partición y búsqueda de hiperparámetros

- **Prueba:** los 12 mapas de 2024.
- **Entrenamiento y validación:** 2016–2023 (80 / 20), con una muestra de entrenamiento de `N_MAX_ENTRENAMIENTO` registros.

La búsqueda paralela (46 combinaciones) ya se corrió y sus tiempos están en `resultados/desempeno_paralelo.csv` y `resultados/estatico_vs_dinamico.csv`. Con `REPETIR_BUSQUEDA = True` se vuelve a correr.

In [ ]:
ent, val, prueba = datos.particionar(df, anio_prueba=ANIO_PRUEBA, n_max_entrenamiento=N_MAX_ENTRENAMIENTO)
X_ent, y_ent = ent[datos.COLUMNAS_X].to_numpy(), ent[datos.COLUMNA_Y].to_numpy()
X_val, y_val = val[datos.COLUMNAS_X].to_numpy(), val[datos.COLUMNA_Y].to_numpy()
X_pru, y_pru = prueba[datos.COLUMNAS_X].to_numpy(), prueba[datos.COLUMNA_Y].to_numpy()

if REPETIR_BUSQUEDA:
    tareas = modelos.lista_de_tareas()
    tiempos, resultados_por_p = {}, {}
    for p in LISTA_PROCESOS:
        res, tiempos[p] = par.busqueda(tareas, (X_ent, y_ent, X_val, y_val), procesos=p, reparto="dinamico")
        resultados_por_p[p] = pd.DataFrame(res)
        print(f"{p:>2} procesos: {tiempos[p]:8.2f} s")
    resultados_por_p[1].sort_values("RMSE").to_csv("resultados/busqueda_hiperparametros.csv", index=False)
    par.metricas_paralelas(tiempos).to_csv("resultados/desempeno_paralelo.csv", index=False)

busqueda = pd.read_csv("resultados/busqueda_hiperparametros.csv")
mejores = busqueda.sort_values("RMSE").groupby("modelo").head(1).set_index("modelo")
display(pd.read_csv("resultados/desempeno_paralelo.csv").round(3))
display(pd.read_csv("resultados/estatico_vs_dinamico.csv").round(3))
mejores[["params", "RMSE", "MAE", "R2"]]

## 4. Evaluación temporal de 2024 contra referencias

Cada modelo se entrena con su mejor combinación y predice los 12 mapas de 2024. Las referencias son:
- **Persistencia t−1:** el mes será igual al mes anterior.
- **Persistencia t−12:** el mes será igual al mismo mes del año anterior.
- **Climatología:** el promedio de ese mes en ese pixel durante 2016–2023.

In [ ]:
import evaluacion as ev

entrenados, tiempo_entrenamiento = {}, {}
for nombre, fila in mejores.iterrows():
    m = modelos.crear_modelo(nombre, ast.literal_eval(fila["params"]))
    t0 = time.perf_counter(); m.fit(X_ent, y_ent); tiempo_entrenamiento[nombre] = time.perf_counter() - t0
    entrenados[nombre] = m
    print(f"{nombre}: entrenado en {tiempo_entrenamiento[nombre]:.1f} s")

prueba = ev.agregar_referencias(prueba, ev.climatologia(df, ANIO_PRUEBA))
for nombre, m in entrenados.items():
    procesos = NUCLEOS if nombre != "RF" else 1     # RF predice rápido; enviar sus 200 árboles a cada proceso no conviene
    prueba[f"pred_{nombre}"], t = par.predecir_paralelo(m, X_pru, procesos=procesos)
    print(f"{nombre}: {len(X_pru):,} pixeles predichos en {t:.1f} s con {procesos} procesos")

METODOS = ["KNN", "RF", "SVR", "Persistencia_t1", "Persistencia_t12", "Climatologia"]
metricas_2024 = ev.tabla_metricas(prueba, METODOS)
metricas_2024.to_csv("resultados/metricas_2024.csv", index=False)
metricas_2024.round(3)

In [ ]:
por_mes = ev.rmse_por_mes(prueba, METODOS)
por_mes.to_csv("resultados/rmse_por_mes_2024.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 3.8))
for n in METODOS:
    estilo = "--" if n not in entrenados else "-"
    ax.plot(por_mes["mes"], por_mes[n], marker="o", lw=2 if n in entrenados else 1.4,
            ls=estilo, color=COLORES[n], label=n.replace("_", " "))
ax.set_ylabel("RMSE (°C)"); ax.set_title("Error de cada método en los 12 meses de 2024")
ax.legend(frameon=False, ncol=3, fontsize=9)
fig.tight_layout(); fig.savefig("figuras/rmse_por_mes_2024.png", dpi=200); plt.show()
por_mes.round(2)

In [ ]:
mejor = metricas_2024.loc[metricas_2024["metodo"].isin(entrenados), "metodo"].iloc[0]
mes_ej = int(por_mes.assign(m=range(1, 13)).loc[por_mes[mejor].idxmax(), "m"])   # el mes más difícil
real = ev.a_mapa(prueba, "LST", mes_ej, FORMA)
pred = ev.a_mapa(prueba, f"pred_{mejor}", mes_ej, FORMA)
lim = np.nanpercentile(np.abs(pred - real), 99)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), layout="constrained")
vmin, vmax = np.nanpercentile(real, [1, 99])
for ax, img, titulo in ((axes[0], real, "Real"), (axes[1], pred, f"Predicho ({mejor})")):
    im = ax.imshow(img, cmap="inferno", vmin=vmin, vmax=vmax, extent=EXTENSION)
    ax.set_title(f"{titulo}, {ev.NOMBRES_MES[mes_ej - 1]} 2024"); ax.grid(False)
fig.colorbar(im, ax=axes[1], label="LST (°C)", shrink=0.85)
im = axes[2].imshow(pred - real, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=EXTENSION)
axes[2].set_title("Error (predicho − real)"); axes[2].grid(False)
fig.colorbar(im, ax=axes[2], label="°C", shrink=0.85)
fig.savefig("figuras/mapas_2024_real_predicho_error.png", dpi=200); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.4))
muestra = prueba.sample(20000, random_state=1)
ax.scatter(muestra["LST"], muestra[f"pred_{mejor}"], s=2, alpha=0.25, color=COLORES[mejor])
lims = [muestra["LST"].min(), muestra["LST"].max()]
ax.plot(lims, lims, ls="--", color="#999999")
ax.set_xlabel("LST real (°C)"); ax.set_ylabel("LST predicha (°C)"); ax.set_title(f"{mejor}, 2024")
fig.tight_layout(); fig.savefig("figuras/real_vs_predicho_2024.png", dpi=200); plt.show()

## 5. Predicción paralela de los mapas de 2024

Predecir los ~490 mil pixel-mes de 2024 es costoso para KNN (buscar vecinos) y SVR (evaluar el kernel contra todos los vectores de soporte). Los pixeles se parten en bloques y cada proceso predice los suyos; el modelo se envía una sola vez a cada proceso. Se verifica que el resultado sea igual al secuencial.

In [ ]:
filas = []
for nombre in ("KNN", "SVR"):
    m = entrenados[nombre]
    tiempos_pred = {}
    for p in LISTA_PROCESOS:
        y_p, tiempos_pred[p] = par.predecir_paralelo(m, X_pru, procesos=p)
        if p == 1:
            y_1 = y_p
        print(f"{nombre} {p:>2} procesos: {tiempos_pred[p]:7.2f} s  | dif. máx. con secuencial = {np.abs(y_p - y_1).max():.1e}")
    filas.append(par.metricas_paralelas(tiempos_pred).assign(modelo=nombre))
prediccion_paralela = pd.concat(filas, ignore_index=True)
prediccion_paralela.to_csv("resultados/prediccion_paralela.csv", index=False)
prediccion_paralela.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for nombre, g in prediccion_paralela.groupby("modelo"):
    axes[0].plot(g["procesos"], g["tiempo_s"], marker="o", lw=2, color=COLORES[nombre], label=nombre)
    axes[1].plot(g["procesos"], g["speedup"], marker="o", lw=2, color=COLORES[nombre], label=nombre)
    axes[2].plot(g["procesos"], g["eficiencia"], marker="o", lw=2, color=COLORES[nombre], label=nombre)
axes[1].plot(LISTA_PROCESOS, LISTA_PROCESOS, ls="--", color="#bbbbbb", label="Ideal")
axes[2].axhline(1, ls="--", color="#bbbbbb"); axes[2].set_ylim(0, 1.1)
for ax, t in zip(axes, ("Tiempo de predicción (s)", "Speedup", "Eficiencia")):
    ax.set_title(t); ax.set_xlabel("Número de procesos"); ax.set_xticks(LISTA_PROCESOS); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig("figuras/prediccion_paralela.png", dpi=200); plt.show()

## 6. Resumen

In [ ]:
print("Mejor modelo en 2024:", mejor)
display(metricas_2024.round(3))
v2 = tabla_v2.set_index("procesos")
print(f"Construcción de la tabla: v1 secuencial {tiempos_v1[1]:.2f} s -> v2 con {LISTA_PROCESOS[-1]} procesos "
      f"{v2.loc[LISTA_PROCESOS[-1], 'tiempo_s']:.2f} s")
prediccion_paralela.loc[prediccion_paralela["procesos"] == LISTA_PROCESOS[-1]].round(3)